# Hillshade PCA of two LiDAR tiles

One basis is fitted to two adjacent 1 m USGS 3DEP LiDAR tiles near Blue Mesa Reservoir, Colorado, and applied to each, so the component images of both tiles share one color scale. The tiles are public domain and are downloaded on the first run (about 240 MB each). Fitting and writing all products takes about a minute and well under 1 GB of memory.

Only the config cell should need editing for other data.

In [ ]:
from pathlib import Path

DATA_DIR = Path("../example_data/USGS_LiDAR")
OUT_DIR = DATA_DIR / "hspca"
TILES = {  # local name: source URL
    "USGS_LiDAR_1.tif": "https://prd-tnm.s3.amazonaws.com/StagedProducts/Elevation/1m/Projects/CO_WestCentral_2019_A19/TIFF/USGS_1M_13_x29y426_CO_WestCentral_2019_A19.tif",
    "USGS_LiDAR_2.tif": "https://prd-tnm.s3.amazonaws.com/StagedProducts/Elevation/1m/Projects/CO_WestCentral_2019_A19/TIFF/USGS_1M_13_x29y427_CO_WestCentral_2019_A19.tif",
}
AZIMUTHS = (0, 45, 90, 135, 180, 225, 270, 315)  # sun azimuths [degrees clockwise from north]
ALTITUDES = (30, 60)                             # sun altitudes [degrees]
Z_UNITS = None          # elevation unit; None means the CRS unit (meters for UTM 13N)
VIEW_TILE = "USGS_LiDAR_2.tif"
VIEW = (4000, 2000, 2500, 2500)  # figure crop: column offset, row offset, width, height [cells]

In [ ]:
import urllib.request

import matplotlib.pyplot as plt
import rasterio
from rasterio.windows import Window

import disco_tif

print("disco-tif", disco_tif.__version__)
DATA_DIR.mkdir(parents=True, exist_ok=True)
dems = []
for name, url in TILES.items():
    path = DATA_DIR / name
    if not path.exists():
        print("downloading", name)
        urllib.request.urlretrieve(url, path)
    dems.append(path)

Fit one basis to both tiles. The moments of the two tiles are added, so the basis is that of all their cells together, without building a mosaic.

In [ ]:
basis = disco_tif.fit_basis(dems, azimuths=AZIMUTHS, altitudes=ALTITUDES, z_units=Z_UNITS)
print("explained variance [%]:", (100 * basis.explained).round(1))
print("weights, rows (east, north, up), columns PC1-PC3:")
print(basis.weights.round(3))

Write the products of each tile under that basis: one uint8 GeoTIFF per component, an RGB composite, and the basis JSON. A single hillshade is written for comparison.

In [ ]:
products = {dem.name: disco_tif.apply_basis(dem, basis, OUT_DIR, z_units=Z_UNITS) for dem in dems}
hillshade = OUT_DIR / f"{Path(VIEW_TILE).stem}_hs315.tif"
disco_tif.hillshade(DATA_DIR / VIEW_TILE, hillshade, azimuth=315, altitude=45, z_units=Z_UNITS)
products[VIEW_TILE]

In [ ]:
window = Window(*VIEW)
view = products[VIEW_TILE]
panels = {"Hillshade, sun at 315/45": hillshade, "PC1": view["c1"], "PC2": view["c2"], "RGB of PC1-PC3": view["rgb"]}

fig, axes = plt.subplots(2, 2, figsize=(12, 12))
for ax, (title, path) in zip(axes.flat, panels.items()):
    with rasterio.open(path) as src:
        image = src.read(window=window).transpose(1, 2, 0).squeeze()
    ax.imshow(image, cmap="gray")
    ax.set_title(title)
    ax.set_axis_off()
fig.tight_layout()
fig.savefig(OUT_DIR / "hspca_example.png", dpi=80)

The tiles were cut from one survey, so the cells along their shared edge get one-sided gradients and differ slightly from a whole read. For one seamless product, run on a GDAL VRT of the tiles instead:

```bash
gdalbuildvrt mosaic.vrt USGS_LiDAR_1.tif USGS_LiDAR_2.tif
disco-tif hspca mosaic.vrt
```